<a href="https://colab.research.google.com/github/Jeyarakavan/Multi-agent-hospital-system/blob/main/Hospital_Call_ASR_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch torchaudio transformers datasets openai-whisper evaluate jiwer soundfile librosa accelerate gtts

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 40.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 4.0 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import soundfile as sf
import librosa
import os
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
from gtts import gTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


In [3]:
print("Creating dataset...")
os.makedirs("medical_calls", exist_ok=True)

conversations = [
    "I need to book an appointment with Dr. Smith for tomorrow morning.",
    "I have severe chest pain and difficulty breathing. It's an emergency!",
    "What are your visiting hours? I want to see my mother.",
    "I need to cancel my appointment for Friday. I have a fever.",
    "Is Dr. Johnson available next Monday? I need a follow-up.",
    "My child has high fever and rash. Can we see a pediatrician today?",
    "I'm calling to reschedule my appointment from tomorrow to next week.",
    "Hello, this is John Smith. My date of birth is 05/15/1980."
]

audio_data = []
for i, text in enumerate(conversations):
    print(f"  Creating sample {i+1}")
    tts = gTTS(text, lang='en', slow=False)
    path = f"medical_calls/call_{i}.wav"
    tts.save(path)
    audio, sr = sf.read(path)
    audio_data.append({"audio": audio, "sr": sr, "text": text})

print(f"✅ Created {len(audio_data)} samples")

Creating dataset...
  Creating sample 1
  Creating sample 2
  Creating sample 3
  Creating sample 4
  Creating sample 5
  Creating sample 6
  Creating sample 7
  Creating sample 8
✅ Created 8 samples


In [4]:
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
print("✅ Processor loaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

✅ Processor loaded


In [5]:
def prepare_data(data):
    processed = []
    for item in data:
        if item['sr'] != 16000:
            audio = librosa.resample(item['audio'], orig_sr=item['sr'], target_sr=16000)
        else:
            audio = item['audio']

        features = processor(audio, sampling_rate=16000, return_tensors="pt").input_features[0]
        labels = processor.tokenizer(item['text'], return_tensors="pt").input_ids[0]

        processed.append({
            "input_features": features.numpy(),
            "labels": labels.numpy()
        })
    return processed

train_data = prepare_data(audio_data[:6])
eval_data = prepare_data(audio_data[6:])
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

Train: 6, Eval: 2


In [6]:
def collator(features):
    input_features = [{"input_features": f["input_features"]} for f in features]
    batch = processor.feature_extractor.pad(input_features, return_tensors="pt")

    label_features = [{"input_ids": f["labels"]} for f in features]
    labels_batch = processor.tokenizer.pad(label_features, return_tensors="pt")
    labels = labels_batch["input_ids"].masked_fill(labels_batch["attention_mask"].ne(1), -100)
    batch["labels"] = labels
    return batch

print("✅ Collator ready")

✅ Collator ready


In [7]:
wer = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    return {"wer": wer.compute(predictions=pred_str, references=label_str)}

In [8]:
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)

args = Seq2SeqTrainingArguments(
    output_dir="./whisper_model",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=5,
    save_steps=5,
    logging_steps=2,
    logging_dir="./logs",
    report_to="none",
    predict_with_generate=True,
    fp16=True if device == "cuda" else False,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

print("✅ Model and args ready")

model.safetensors:   0%|          | 0.00/151M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ Model and args ready


In [10]:
# CELL 9: Create Trainer (FIXED)
trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    data_collator=collator,
    compute_metrics=compute_metrics,

)

print(f"Trainer ready - Training: {len(train_data)} samples")

Trainer ready - Training: 6 samples


In [11]:
print("🚀 Training started...")
trainer.train()
print("✅ Training complete!")

🚀 Training started...


Step,Training Loss,Validation Loss,Wer
5,4.200931,3.933868,0.136364


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['proj_out.weight'].


✅ Training complete!


In [13]:
model.save_pretrained("./final_speech_agent")
processor.save_pretrained("./final_speech_agent")
print("✅ Model saved to ./final_speech_agent")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to ./final_speech_agent


In [14]:
test_model = WhisperForConditionalGeneration.from_pretrained("./final_speech_agent").to(device)
test_processor = WhisperProcessor.from_pretrained("./final_speech_agent")

test_audio = audio_data[6]['audio']
if audio_data[6]['sr'] != 16000:
    test_audio = librosa.resample(test_audio, orig_sr=audio_data[6]['sr'], target_sr=16000)

inputs = test_processor(test_audio, sampling_rate=16000, return_tensors="pt").to(device)
with torch.no_grad():
    pred_ids = test_model.generate(inputs.input_features)

print(f"\nOriginal: {audio_data[6]['text']}")
print(f"Transcribed: {test_processor.batch_decode(pred_ids, skip_special_tokens=True)[0]}")

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]


Original: I'm calling to reschedule my appointment from tomorrow to next week.
Transcribed:  I'm calling to reschedule my appointment from tomorrow to next week.


In [15]:
!zip -r speech_agent_model.zip final_speech_agent
print("✅ speech_agent_model.zip created - Download from Files panel")

  adding: final_speech_agent/ (stored 0%)
  adding: final_speech_agent/generation_config.json (deflated 70%)
  adding: final_speech_agent/config.json (deflated 60%)
  adding: final_speech_agent/tokenizer.json (deflated 82%)
  adding: final_speech_agent/tokenizer_config.json (deflated 74%)
  adding: final_speech_agent/model.safetensors (deflated 42%)
  adding: final_speech_agent/processor_config.json (deflated 48%)
✅ speech_agent_model.zip created - Download from Files panel
